# 🔵 Unidad 3 · Clase 3 — Aprendizaje No Supervisado (Clustering)
### K-Means · Segmentación · PCA · Reducción de dimensiones

---

## ¿De dónde venimos?

En las clases anteriores entrenamos modelos **supervisados**: le dábamos al algoritmo miles de ejemplos con sus respuestas conocidas (aprueba = 1 / reprueba = 0) y él aprendía a predecir esas respuestas.

Hoy cambiamos completamente el paradigma.

---

## El giro fundamental: aprender sin respuestas

Imagina que eres director académico y recibes los datos de 1 000 estudiantes: horas de estudio, sueño, redes sociales, salud mental, dieta...

**Pero nadie te dijo quién aprueba y quién reprueba.**

¿Puedes encontrar grupos de estudiantes con comportamientos similares? ¿Existen perfiles naturales — el "estudiante dedicado", el "estudiante desorganizado", el "estudiante quemado" — sin que nadie te los defina?

Eso es exactamente lo que hace el **Aprendizaje No Supervisado**.

---

## Supervisado vs No Supervisado

| | Supervisado | No Supervisado |
|---|---|---|
| **¿Tiene etiquetas?** | ✅ Sí (aprueba = 0/1) | ❌ No — el algoritmo explora solo |
| **¿Qué aprende?** | A predecir una respuesta conocida | A encontrar estructura oculta en los datos |
| **Pregunta que responde** | ¿Este estudiante aprueba? | ¿Qué tipos de estudiantes existen? |
| **Algoritmos típicos** | Regresión, Árbol, Random Forest | K-Means, DBSCAN, PCA |
| **Evaluación** | Accuracy, F1, R² | Silhouette score, inercia, interpretación |

---

## El experimento de hoy

Vamos a hacer algo pedagógicamente muy poderoso:

1. **Quitamos** la columna `aprueba` del dataset — el algoritmo no sabrá quién aprueba
2. Dejamos que **K-Means encuentre grupos** solo con los hábitos de los estudiantes
3. Al final, **revelamos** si los grupos que encontró corresponden con quienes aprueban y reprueban

Si K-Means —sin haber visto nunca las etiquetas— logra separar grupos que se alinean con aprobados/reprobados, eso prueba que **los hábitos son suficientes para predecir el resultado académico**, incluso sin un modelo supervisado.

---

## ¿Qué construiremos hoy?

| Herramienta | Para qué |
|---|---|
| **K-Means** | Encontrar K grupos de estudiantes con hábitos similares |
| **Método del Codo** | Elegir el número óptimo de clusters K |
| **Silhouette Score** | Medir qué tan bien definidos están los clusters |
| **PCA** | Reducir 14 dimensiones a 2 para poder visualizar los clusters |

> ⏱️ Duración estimada: **~2 horas**

In [ ]:
import pandas as pd                      # DataFrames: leer, filtrar y manipular tablas de datos
import numpy as np                       # arrays y operaciones matemáticas vectorizadas eficientes
import matplotlib.pyplot as plt          # motor base para construir gráficas personalizadas
import seaborn as sns                    # gráficas estadísticas de alto nivel con menos código

# ── Clustering ────────────────────────────────────────────────────────────────
# KMeans: implementa el algoritmo K-Means; agrupa datos en K clusters
# minimizando la distancia de cada punto a su centroide más cercano
from sklearn.cluster import KMeans

# silhouette_score: métrica de calidad del clustering (rango -1 a +1)
# mide qué tan compactos y bien separados están los clusters entre sí
from sklearn.metrics import silhouette_score

# ── Reducción de dimensiones ─────────────────────────────────────────────────
# PCA (Principal Component Analysis): reduce N features correlacionadas
# a M componentes ortogonales que capturan la mayor varianza posible
# Lo usamos para comprimir 14 features a 2 dimensiones y poder graficar
from sklearn.decomposition import PCA

# ── Preprocesamiento ─────────────────────────────────────────────────────────
# StandardScaler: transforma cada feature a media=0 y std=1
# CRÍTICO para K-Means: sin escalar, features con números grandes dominarían
# injustamente el cálculo de distancias euclidianas
# OrdinalEncoder: convierte categorías con orden real (Poor=0, Fair=1, Good=2)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

# warnings: librería estándar de Python para controlar mensajes de advertencia
# filterwarnings('ignore') suprime advertencias menores que no afectan el resultado
import warnings
warnings.filterwarnings('ignore')

---
## 📦 Sección 1 — Importaciones

Cargamos todas las librerías al inicio de la sesión. Hoy incorporamos herramientas completamente nuevas que no habíamos visto en las clases anteriores: `KMeans` para clustering, `PCA` para reducción de dimensiones y `silhouette_score` para medir la calidad de los grupos encontrados.

In [ ]:
# pd.read_csv lee el archivo CSV y lo convierte en un DataFrame en memoria RAM
df = pd.read_csv('student_habits_performance.csv')

# student_id es un identificador único sin valor predictivo para los hábitos
# Si K-Means lo usara, agruparía estudiantes por número de ID en vez de por comportamiento
df = df.drop(columns=['student_id'])

# ── Guardar las etiquetas reales ANTES de que el modelo las vea ───────────────
# Guardamos exam_score y aprueba ahora, antes de eliminarlas
# Las usaremos SOLO al final (Sección 7) para validar si K-Means descubrió la estructura real
# Esto simula un escenario real donde el modelo no tiene acceso a las respuestas
exam_score_real = df['exam_score'].copy()              # puntaje numérico de cada estudiante
aprueba_real    = (df['exam_score'] >= 60).astype(int) # 1=aprueba, 0=reprueba

# Eliminamos las columnas target: el modelo NO puede ver esta información
# Si K-Means viera exam_score, trivialmente agruparía por puntaje en vez de aprender de hábitos
df = df.drop(columns=['exam_score'])

# ── Tratar nulos ─────────────────────────────────────────────────────────────
# parental_education_level tiene 91 NaN (~9%); imputamos con la moda para no perder filas
# .mode() devuelve los valores más frecuentes como Serie; [0] toma el primero
moda_edu = df['parental_education_level'].mode()[0]
df['parental_education_level'] = df['parental_education_level'].fillna(moda_edu)

# ── Encoding de variables categóricas ────────────────────────────────────────
# Todas las features deben ser numéricas para que K-Means calcule distancias euclidianas

# OrdinalEncoder para diet_quality: respeta el orden real Poor(0) < Fair(1) < Good(2)
# Si usáramos LabelEncoder automático el orden sería alfabético (incorrecto)
oe_diet = OrdinalEncoder(categories=[['Poor', 'Fair', 'Good']])
# fit_transform: aprende las categorías y las convierte a enteros en un paso
# Devuelve array 2D (1000,1); reshape(-1) lo aplana a vector 1D (1000,)
df['diet_quality'] = oe_diet.fit_transform(df[['diet_quality']]).reshape(-1)

# OrdinalEncoder para internet_quality: Poor(0) < Average(1) < Good(2)
oe_net = OrdinalEncoder(categories=[['Poor', 'Average', 'Good']])
df['internet_quality'] = oe_net.fit_transform(df[['internet_quality']]).reshape(-1)

# OrdinalEncoder para parental_education_level: High School(0) < Bachelor(1) < Master(2)
oe_edu = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master']])
df['parental_education_level'] = oe_edu.fit_transform(df[['parental_education_level']]).reshape(-1)

# Encoding binario: Yes → 1, No → 0 (comparación directa, más simple que OrdinalEncoder)
df['part_time_job'] = (df['part_time_job'] == 'Yes').astype(int)
df['extracurricular_participation'] = (df['extracurricular_participation'] == 'Yes').astype(int)

# One-Hot Encoding para gender: nominal (sin orden), más de dos categorías
# drop_first=True elimina la categoría 'Female' para evitar multicolinealidad
df = pd.get_dummies(df, columns=['gender'], drop_first=True, dtype=int)

# ── Escalado: paso CRÍTICO para K-Means ──────────────────────────────────────
# K-Means calcula distancias euclidianas: ||punto_a - punto_b||
# Sin escalar, attendance_percentage (rango 56-100) domina sobre part_time_job (rango 0-1)
# StandardScaler hace que TODAS las features contribuyan equitativamente a las distancias
scaler = StandardScaler()

# fit_transform sobre TODOS los datos (no hay train/test en clustering)
# El scaler aprende media y std de los 1000 estudiantes y transforma
# El resultado es un array NumPy con media≈0 y std≈1 en cada columna
X_scaled = scaler.fit_transform(df)

# Verificación: después del escalado cada columna debe tener media≈0 y std≈1
print(f'Dataset listo para clustering:')
print(f'  Estudiantes: {X_scaled.shape[0]}  |  Features: {X_scaled.shape[1]}')
print(f'  Media global después de escalar: {X_scaled.mean():.6f}  (debe ser ≈ 0)')
print(f'  Std global después de escalar:   {X_scaled.std():.6f}   (debe ser ≈ 1)')

In [ ]:
# Carga el CSV desde disco y lo convierte en DataFrame de pandas
df = pd.read_csv('student_habits_performance.csv')

# Eliminamos student_id: identificador único, no tiene poder informativo sobre los hábitos
df = df.drop(columns=['student_id'])

# Guardamos exam_score y calculamos aprueba AHORA, antes de que el modelo los vea
# Los usaremos solo al final para validar si K-Means encontró los grupos correctos
# Esta es la "respuesta" que ocultaremos al algoritmo durante todo el experimento
exam_score_real = df['exam_score'].copy()            # puntaje real de cada estudiante
aprueba_real    = (df['exam_score'] >= 60).astype(int)  # etiqueta real (0 o 1)

# Eliminamos las columnas de target: el modelo NO puede ver estas variables
# Si K-Means viera exam_score, trivialmente agruparia por puntaje — no aprendería patrones de hábitos
df = df.drop(columns=['exam_score'])

# ── Tratar nulos ─────────────────────────────────────────────────────────────
# parental_education_level tiene 91 NaN (~9%)
# Los imputamos con la moda para no perder esas filas
moda_edu = df['parental_education_level'].mode()[0]   # valor más frecuente
df['parental_education_level'] = df['parental_education_level'].fillna(moda_edu)

# ── Encoding de variables categóricas ────────────────────────────────────────
# Todas las variables deben ser numéricas para que K-Means pueda calcular distancias

# OrdinalEncoder para variables con orden real (Poor=0, Fair=1, Good=2)
oe_diet = OrdinalEncoder(categories=[['Poor', 'Fair', 'Good']])
df['diet_quality'] = oe_diet.fit_transform(df[['diet_quality']]).reshape(-1)

oe_net = OrdinalEncoder(categories=[['Poor', 'Average', 'Good']])
df['internet_quality'] = oe_net.fit_transform(df[['internet_quality']]).reshape(-1)

oe_edu = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master']])
df['parental_education_level'] = oe_edu.fit_transform(df[['parental_education_level']]).reshape(-1)

# Encoding binario para variables Yes/No → 0/1
df['part_time_job'] = (df['part_time_job'] == 'Yes').astype(int)
df['extracurricular_participation'] = (df['extracurricular_participation'] == 'Yes').astype(int)

# One-Hot Encoding para gender (nominal, sin orden)
df = pd.get_dummies(df, columns=['gender'], drop_first=True, dtype=int)

# ── Escalar todas las features ────────────────────────────────────────────────
# En clustering NO hay train/test split — usamos TODOS los datos
# El scaler aprende media y std de los 1000 estudiantes y los transforma
scaler = StandardScaler()

# fit_transform: aprende los parámetros y transforma en un solo paso
# El resultado es un array NumPy (no DataFrame); lo convertimos de vuelta con columnas
X_scaled = scaler.fit_transform(df)

# Verificamos: después del escalado, cada columna debe tener media≈0 y std≈1
print(f'Dataset listo para clustering:')
print(f'  Estudiantes: {X_scaled.shape[0]}')
print(f'  Features:    {X_scaled.shape[1]}')
print(f'  Media de todas las features después de escalar: {X_scaled.mean():.6f}  (debe ser ≈ 0)')
print(f'  Std de todas las features después de escalar:   {X_scaled.std():.6f}   (debe ser ≈ 1)')

In [ ]:
# Evaluamos K-Means para K desde 1 hasta 10 clusters
k_valores   = range(1, 11)   # valores de K a probar
inercias    = []             # inercia (WCSS) de cada K — para el método del codo
silhouettes = []             # silhouette score de cada K — para validar la calidad

for k in k_valores:
    # KMeans instancia un modelo con k clusters
    # n_init=10: lanza 10 inicializaciones aleatorias distintas y conserva la mejor
    #   (la que tenga menor inercia) — evita quedarse en mínimos locales
    # max_iter=300: número máximo de iteraciones antes de parar si no convergió
    # random_state=42: fija la semilla para reproducibilidad del proceso aleatorio
    kmeans_k = KMeans(n_clusters=k, n_init=10, max_iter=300, random_state=42)

    # fit_predict: entrena K-Means y devuelve la etiqueta de cluster de cada punto
    # labels_k es un array de shape (1000,) con valores 0, 1, ..., k-1
    labels_k = kmeans_k.fit_predict(X_scaled)

    # .inertia_: suma de distancias al cuadrado de cada punto a su centroide asignado
    # Con K=1 es máxima (todos en un grupo); con K=N es 0 (cada punto es su propio cluster)
    inercias.append(kmeans_k.inertia_)

    # silhouette_score: no tiene sentido para K=1 (solo hay un grupo, no hay "separación")
    # Para K>=2: mide cohesión interna vs separación externa de los clusters
    if k >= 2:
        silhouettes.append(silhouette_score(X_scaled, labels_k))
    else:
        silhouettes.append(None)   # marcador para K=1 (sin valor válido)

# Tabla resumen con inercia y silhouette score de cada K
print('K | Inercia      | Silhouette')
print('-' * 35)
for k, ine, sil in zip(k_valores, inercias, silhouettes):
    sil_str = f'{sil:.4f}' if sil is not None else '  N/A  '
    print(f'{k:2} | {ine:10.1f}  | {sil_str}')

In [ ]:
# Creamos una figura con 2 paneles lado a lado para comparar las dos métricas
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Panel izquierdo: Método del Codo (Inercia vs K) ──────────────────────────
# La curva empieza alta (K=1, todos en un grupo) y baja al agregar más clusters
# El "codo" es el punto donde la curva deja de bajar bruscamente → K óptimo
axes[0].plot(k_valores, inercias, marker='o', color='steelblue', linewidth=2, markersize=6)
axes[0].set_title(
    'Método del Codo — Inercia vs K
'
    'Busca el "codo": donde agregar más clusters ya no reduce mucho la inercia'
)
axes[0].set_xlabel('Número de clusters K')
axes[0].set_ylabel('Inercia (suma de distancias² dentro de cada cluster)')
axes[0].grid(alpha=0.3)

# ── Panel derecho: Silhouette Score vs K ─────────────────────────────────────
# Solo graficamos desde K=2 porque para K=1 no hay separación que medir
k_vals_sil = list(k_valores)[1:]     # K de 2 a 10
sil_vals   = silhouettes[1:]         # scores correspondientes (sin el None de K=1)

# Un score más alto indica clusters más compactos y mejor separados entre sí
axes[1].plot(k_vals_sil, sil_vals, marker='s', color='seagreen', linewidth=2, markersize=6)
axes[1].set_title(
    'Silhouette Score vs K
'
    'Mayor score = clusters más compactos e internamente coherentes'
)
axes[1].set_xlabel('Número de clusters K')
axes[1].set_ylabel('Silhouette Score (−1 = malo, +1 = excelente)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Elegimos K=3 como punto óptimo basado en el análisis del codo y silhouette
# K=3 suele capturar 3 perfiles naturales en datos de estudiantes:
#   - Grupo con hábitos excelentes (estudia mucho, duerme bien, buena salud mental)
#   - Grupo con hábitos intermedios (el "promedio")
#   - Grupo con hábitos problemáticos (poco estudio, muchas redes, mal descanso)
K_OPTIMO = 3

# Instancia el modelo K-Means con el número óptimo de clusters
kmeans = KMeans(
    n_clusters=K_OPTIMO,   # número de grupos a encontrar en los datos
    n_init=10,             # 10 inicializaciones aleatorias → nos quedamos con la mejor
    max_iter=300,          # máximo de iteraciones antes de parar si no convergió
    random_state=42        # semilla para reproducibilidad del proceso de inicialización
)

# fit_predict: entrena K-Means (iteraciones hasta convergencia) y asigna etiquetas
# Devuelve array (1000,) con el número de cluster de cada estudiante: [0, 2, 1, 0, ...]
etiquetas_cluster = kmeans.fit_predict(X_scaled)

# Agrega la etiqueta de cluster al DataFrame para facilitar el análisis de perfiles
df['cluster'] = etiquetas_cluster

# .inertia_: inercia final después de la convergencia con el K óptimo
# silhouette_score: qué tan bien definidos están los clusters con K=3
# .n_iter_: cuántas iteraciones necesitó el algoritmo para converger
print(f'K-Means entrenado con K={K_OPTIMO} clusters')
print(f'Inercia final:                  {kmeans.inertia_:.2f}')
print(f'Silhouette Score:               {silhouette_score(X_scaled, etiquetas_cluster):.4f}')
print(f'Iteraciones hasta convergencia: {kmeans.n_iter_}')
print()
print('Distribución de estudiantes por cluster:')
for k in range(K_OPTIMO):
    n = (etiquetas_cluster == k).sum()     # cuenta cuántos estudiantes pertenecen al cluster k
    pct = n / len(etiquetas_cluster)       # proporción del total
    print(f'  Cluster {k}: {n} estudiantes ({pct:.1%})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Panel izquierdo: Método del Codo (Elbow Method) ──────────────────────────
# Graficamos la inercia vs el número de clusters
# El "codo" de la curva (donde deja de bajar bruscamente) indica el K óptimo
axes[0].plot(k_valores, inercias, marker='o', color='steelblue', linewidth=2, markersize=6)

axes[0].set_title(
    'Método del Codo — Inercia vs K
'
    'El codo de la curva señala el K donde agregar más clusters no compensa'
)
axes[0].set_xlabel('Número de clusters K')
axes[0].set_ylabel('Inercia (suma de distancias al cuadrado)')
axes[0].grid(alpha=0.3)

# ── Panel derecho: Silhouette Score ──────────────────────────────────────────
# El Silhouette Score mide qué tan bien están separados los clusters
# Rango: -1 (mal) a +1 (excelente). Queremos el K con mayor silhouette score
# Solo graficamos desde K=2 (para K=1 no hay separación que medir)
k_vals_sil = list(k_valores)[1:]     # K desde 2 hasta 10
sil_vals   = silhouettes[1:]         # scores desde K=2

axes[1].plot(k_vals_sil, sil_vals, marker='s', color='seagreen', linewidth=2, markersize=6)

axes[1].set_title(
    'Silhouette Score vs K
'
    'Mayor score = clusters más compactos y mejor separados entre sí'
)
axes[1].set_xlabel('Número de clusters K')
axes[1].set_ylabel('Silhouette Score (−1 a +1)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# PCA con todos los componentes posibles (n_components=14, igual que el número de features)
# Lo hacemos completo primero para ver cuánta varianza captura cada componente
# y decidir con cuántos componentes queremos quedarnos
pca_completo = PCA(n_components=14, random_state=42)

# fit_transform: encuentra las 14 direcciones de máxima varianza (fit)
# y proyecta los datos en ese espacio (transform)
# Descartamos el resultado: solo nos interesa el análisis de varianza, no los datos proyectados
pca_completo.fit_transform(X_scaled)

# explained_variance_ratio_: array con la proporción de varianza explicada por cada componente
# El primer componente siempre captura más varianza que el segundo, y así sucesivamente
varianza_por_componente = pca_completo.explained_variance_ratio_

# np.cumsum calcula la suma acumulativa: [v1, v1+v2, v1+v2+v3, ...]
# Nos dice cuánta varianza total tenemos con 1, 2, 3... componentes
varianza_acumulada = np.cumsum(varianza_por_componente)

# Creamos dos paneles para visualizar la varianza individual y la acumulada
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Panel izquierdo: varianza por componente (barras) ────────────────────────
componentes_idx = range(1, 15)   # etiquetas del eje X: 1, 2, ..., 14
# varianza_por_componente * 100: convierte proporción (0-1) a porcentaje (0-100)
axes[0].bar(componentes_idx, varianza_por_componente * 100,
            color='steelblue', edgecolor='white')
axes[0].set_title(
    'Varianza explicada por cada componente principal
'
    'Los primeros componentes capturan la mayor parte de la información'
)
axes[0].set_xlabel('Componente Principal (PC)')
axes[0].set_ylabel('Varianza explicada (%)')
axes[0].set_xticks(componentes_idx)   # muestra todos los números 1-14 en el eje X

# ── Panel derecho: varianza acumulada (línea) ─────────────────────────────────
axes[1].plot(componentes_idx, varianza_acumulada * 100,
             marker='o', color='seagreen', linewidth=2, markersize=5)

# Líneas de referencia horizontales: umbrales comunes para decidir cuántos PCs usar
axes[1].axhline(80, color='red',    linestyle='--', linewidth=1, label='80% varianza')
axes[1].axhline(95, color='orange', linestyle='--', linewidth=1, label='95% varianza')

axes[1].set_title(
    'Varianza acumulada vs número de componentes
'
    'Elegir el mínimo de componentes que supere el umbral deseado (80% o 95%)'
)
axes[1].set_xlabel('Número de componentes principales')
axes[1].set_ylabel('Varianza acumulada (%)')
axes[1].legend()
axes[1].set_xticks(componentes_idx)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla de varianza para los primeros 6 componentes
print('Varianza explicada por los primeros componentes:')
for i, (v, va) in enumerate(zip(varianza_por_componente[:6], varianza_acumulada[:6]), 1):
    print(f'  PC{i}: {v:.1%} individual  |  {va:.1%} acumulada')

In [ ]:
# PCA con exactamente 2 componentes: proyectamos de 14D a 2D para visualización
# Con 2 componentes perdemos información pero ganamos la capacidad de graficar
pca_2d = PCA(n_components=2, random_state=42)

# fit_transform: aprende las 2 direcciones de máxima varianza (fit) y proyecta (transform)
# X_pca tiene shape (1000, 2): cada fila es un estudiante en el espacio 2D comprimido
# PC1 (columna 0) es el eje X del gráfico; PC2 (columna 1) es el eje Y
X_pca = pca_2d.fit_transform(X_scaled)

# .sum() sobre explained_variance_ratio_ da la varianza total capturada por los 2 componentes
# Un valor de 0.40 significa que los 2 PCs capturan el 40% de la info original
varianza_2d = pca_2d.explained_variance_ratio_.sum()

print(f'Varianza total capturada con 2 componentes: {varianza_2d:.1%}')
print(f'Información "perdida" en la compresión:     {(1 - varianza_2d):.1%}')
print()
# La varianza de cada componente por separado
print(f'PC1 explica: {pca_2d.explained_variance_ratio_[0]:.1%} de la varianza total')
print(f'PC2 explica: {pca_2d.explained_variance_ratio_[1]:.1%} de la varianza total')
print()
print(f'Los 1000 estudiantes ahora viven en un espacio 2D: X_pca.shape = {X_pca.shape}')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Colores distintivos para los 3 clusters: rojo, azul, verde
colores_cluster = ['#e74c3c', '#3498db', '#2ecc71']

# Nombres de los clusters para la leyenda del gráfico
nombres_cluster = [f'Cluster {k}' for k in range(K_OPTIMO)]

# Graficamos cada cluster por separado para que tenga su propia entrada en la leyenda
for k in range(K_OPTIMO):
    # Máscara booleana: True para los estudiantes que pertenecen al cluster k
    mascara = etiquetas_cluster == k

    # X_pca[mascara, 0]: coordenada PC1 (eje X) de los estudiantes del cluster k
    # X_pca[mascara, 1]: coordenada PC2 (eje Y) de los estudiantes del cluster k
    ax.scatter(
        X_pca[mascara, 0],       # posición horizontal (PC1)
        X_pca[mascara, 1],       # posición vertical (PC2)
        c=colores_cluster[k],    # color asignado a este cluster
        label=nombres_cluster[k],# texto que aparecerá en la leyenda
        alpha=0.6,               # transparencia: permite ver puntos superpuestos
        s=30,                    # tamaño de cada punto en puntos cuadrados
        edgecolors='white',      # borde blanco: separa visualmente puntos solapados
        linewidths=0.3           # grosor del borde blanco
    )

# Graficamos los centroides del K-Means proyectados al espacio 2D
# kmeans.cluster_centers_: array (K, 14) — centroides en el espacio original de 14 features
# pca_2d.transform(): proyecta esos centroides al espacio 2D de PCA para ubicarlos en el gráfico
centroides_2d = pca_2d.transform(kmeans.cluster_centers_)

ax.scatter(
    centroides_2d[:, 0],    # coordenada PC1 de cada centroide
    centroides_2d[:, 1],    # coordenada PC2 de cada centroide
    c='black',              # negro: contrasta con los colores de los clusters
    marker='X',             # símbolo X: convención visual para centroides en clustering
    s=200,                  # más grandes que los puntos normales para que destaquen
    zorder=5,               # se dibuja encima de todos los demás elementos
    label='Centroides'
)

ax.set_title(
    f'Clusters de estudiantes — espacio PCA 2D
'
    f'K-Means (K={K_OPTIMO}) | {varianza_2d:.0%} de la varianza original capturada',
    fontsize=12
)
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} varianza)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} varianza)')
ax.legend()
plt.tight_layout()
plt.show()

### 4.1 — ¿Cuánta varianza explica cada componente?

Antes de reducir a 2D, analizamos cuánta información retiene cada componente principal. Esto nos dice cuánta información perdemos al comprimir las 14 features.

In [ ]:
# Features que usaremos para describir los perfiles de cada cluster
# Usamos las features interpretables (antes del OHE de gender) para que el análisis sea legible
features_analisis = ['study_hours_per_day', 'social_media_hours', 'netflix_hours',
                     'attendance_percentage', 'sleep_hours', 'exercise_frequency',
                     'mental_health_rating', 'part_time_job', 'diet_quality',
                     'internet_quality', 'parental_education_level',
                     'extracurricular_participation']

# Construimos un DataFrame auxiliar con las features de análisis y la etiqueta de cluster
df_perfil = df[features_analisis].copy()
df_perfil['cluster'] = etiquetas_cluster   # agrega la columna de cluster para poder agrupar

# groupby('cluster'): agrupa filas por su valor de cluster (0, 1, 2)
# [features_analisis]: selecciona solo las columnas de features (excluye 'cluster')
# .mean(): calcula el promedio de cada feature dentro de cada grupo
# .round(3): redondea a 3 decimales para legibilidad
perfiles = df_perfil.groupby('cluster')[features_analisis].mean().round(3)

# Calculamos también la media global para comparar cada cluster con el promedio general
media_global = df_perfil[features_analisis].mean()

# .T transpone la tabla: features en filas, clusters en columnas
# Facilita la lectura: en cada columna ves todas las características del cluster
print('=== Perfil promedio de cada cluster (valores originales, no escalados) ===')
perfiles.T

In [ ]:
# Normalización min-max de cada feature al rango [0, 1] para comparar en la misma escala
# Sin normalizar, attendance_percentage (valores 56-100) dominaría visualmente sobre
# part_time_job (valores 0-1), aunque ambas puedan ser igualmente importantes
# Fórmula: (valor - mínimo) / (máximo - mínimo) → coloca todos los valores en [0, 1]
perfiles_norm = (perfiles - perfiles.min()) / (perfiles.max() - perfiles.min())

fig, ax = plt.subplots(figsize=(10, 5))

sns.heatmap(
    perfiles_norm.T,        # transponemos: features en filas (eje Y), clusters en columnas (eje X)
    annot=True,             # muestra el valor normalizado [0-1] dentro de cada celda
    fmt='.2f',              # 2 decimales: suficiente precisión sin saturar el texto
    cmap='RdYlGn',          # rojo=valor bajo, amarillo=intermedio, verde=valor alto
    linewidths=0.5,         # líneas divisorias entre celdas para mejorar la legibilidad
    ax=ax
)
ax.set_title(
    'Perfil normalizado de cada cluster — escala [0, 1]
'
    'Verde = valor alto de esa feature en el cluster | Rojo = valor bajo',
    fontsize=12
)
ax.set_xlabel('Cluster')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
print('=== Rasgos más diferenciadores de cada cluster (vs la media global) ===')
print()

for k in range(K_OPTIMO):
    # Cuenta cuántos estudiantes pertenecen al cluster k
    n_estudiantes = (etiquetas_cluster == k).sum()
    print(f'--- Cluster {k} ({n_estudiantes} estudiantes) ---')

    # Diferencia entre el perfil promedio del cluster y la media global del dataset
    # Valor positivo: esa feature es más alta que la media global → rasgo predominante
    # Valor negativo: esa feature es más baja que la media global → rasgo ausente
    diferencias = perfiles.loc[k] - media_global

    # nlargest(3): las 3 features donde este cluster está por encima del promedio global
    top_altas = diferencias.nlargest(3)
    print(f'  Rasgos altos (sobre la media): '
          f'{", ".join([f"{feat} (+{val:.2f})" for feat, val in top_altas.items()])}')

    # nsmallest(3): las 3 features donde este cluster está por debajo del promedio global
    top_bajas = diferencias.nsmallest(3)
    print(f'  Rasgos bajos (bajo la media):  '
          f'{", ".join([f"{feat} ({val:.2f})" for feat, val in top_bajas.items()])}')
    print()

---
## 🎨 Sección 5 — Visualización de los clusters en 2D

Ahora combinamos los dos pasos:
- **K-Means** nos dio la etiqueta de cluster de cada estudiante (0, 1 o 2)
- **PCA** nos dio las coordenadas 2D de cada estudiante

Graficamos los 1 000 estudiantes en 2D, coloreados por su cluster.

In [ ]:
# Construimos un DataFrame auxiliar para la revelación
# Combinamos la etiqueta de cluster (del modelo) con las etiquetas reales (que ocultamos al inicio)
df_revelacion = df[['cluster']].copy()
df_revelacion['aprueba_real']    = aprueba_real.values      # etiqueta real: 0=reprueba, 1=aprueba
df_revelacion['exam_score_real'] = exam_score_real.values   # puntaje numérico real

# pd.crosstab: tabla de contingencia que cruza dos variables categóricas
# Filas: número de cluster (0, 1, 2) | Columnas: si aprueba o no (0 o 1)
# margins=True: agrega totales por fila (Total por cluster) y por columna (Total por clase)
tabla_cruzada = pd.crosstab(
    df_revelacion['cluster'],      # variable de fila
    df_revelacion['aprueba_real'], # variable de columna
    margins=True                   # agrega fila y columna de totales
)
# Renombramos para que sea más legible
tabla_cruzada.columns = ['Reprueba (0)', 'Aprueba (1)', 'Total']
tabla_cruzada.index   = [f'Cluster {i}' for i in range(K_OPTIMO)] + ['Total']

print('=== Tabla cruzada: Cluster asignado vs Resultado real ===')
print(tabla_cruzada)
print()

# Calculamos la tasa de aprobación y el puntaje promedio dentro de cada cluster
# Esto revela si K-Means separó a los estudiantes por rendimiento sin verlo
print('=== Tasa de aprobación y puntaje promedio por cluster ===')
for k in range(K_OPTIMO):
    mask            = df_revelacion['cluster'] == k    # máscara del cluster k
    tasa_aprobacion = df_revelacion.loc[mask, 'aprueba_real'].mean()     # proporción que aprueba
    puntaje_medio   = df_revelacion.loc[mask, 'exam_score_real'].mean()  # puntaje promedio
    n               = mask.sum()   # número de estudiantes en este cluster
    print(f'  Cluster {k} ({n} estudiantes): '
          f'{tasa_aprobacion:.1%} aprueba | puntaje promedio: {puntaje_medio:.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Panel izquierdo: distribución de puntajes por cluster (KDE) ───────────────
# KDE (Kernel Density Estimate): curva de densidad suavizada
# Mejor que histogramas superpuestos para comparar 3 grupos al mismo tiempo
colores_cluster = ['#e74c3c', '#3498db', '#2ecc71']

for k in range(K_OPTIMO):
    # Filtramos los puntajes reales de los estudiantes del cluster k
    mask = df_revelacion['cluster'] == k
    scores_k = df_revelacion.loc[mask, 'exam_score_real']

    # .plot.kde(): dibuja la curva de densidad de los puntajes de este cluster
    # Si los clusters son distintos, las curvas deberían separarse claramente
    scores_k.plot.kde(ax=axes[0], label=f'Cluster {k}',
                      color=colores_cluster[k], linewidth=2)

# Línea vertical en 60: el umbral de aprobación que definimos al inicio
axes[0].axvline(60, color='black', linestyle='--', linewidth=1.5,
                label='Umbral de aprobación (60)')
axes[0].set_title(
    'Distribución de puntajes reales por cluster
'
    'Si los clusters capturan el rendimiento, las curvas deben separarse'
)
axes[0].set_xlabel('Puntaje del examen (exam_score)')
axes[0].set_ylabel('Densidad')
axes[0].legend()

# ── Panel derecho: mismos puntos PCA, coloreados por resultado REAL ───────────
# Comparamos el coloreado por cluster (K-Means) vs el coloreado por la realidad
# Si son similares → K-Means descubrió la estructura real sin ver las etiquetas
colores_real = ['tomato' if a == 0 else 'steelblue'
                for a in df_revelacion['aprueba_real']]

# scatter: un punto por estudiante, coloreado según si realmente aprueba o no
axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                c=colores_real, alpha=0.5, s=20,
                edgecolors='white', linewidths=0.2)

# Creamos la leyenda manualmente porque usamos una lista de colores (no 'hue')
from matplotlib.patches import Patch
leyenda = [Patch(color='tomato',    label='Reprueba (real)'),
           Patch(color='steelblue', label='Aprueba (real)')]
axes[1].legend(handles=leyenda)

axes[1].set_title(
    'Mismos estudiantes en PCA 2D
coloreados por resultado REAL (aprueba/reprueba)
'
    'Compara este gráfico con el de clusters: ¿coincide la separación?'
)
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} varianza)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} varianza)')

plt.tight_layout()
plt.show()

In [ ]:
# Lista de features numéricas originales (antes del OHE) para el análisis de perfiles
features_analisis = ['study_hours_per_day', 'social_media_hours', 'netflix_hours',
                     'attendance_percentage', 'sleep_hours', 'exercise_frequency',
                     'mental_health_rating', 'part_time_job', 'diet_quality',
                     'internet_quality', 'parental_education_level',
                     'extracurricular_participation']

# Agregamos la etiqueta de cluster al DataFrame para poder agrupar por ella
df_perfil = df[features_analisis].copy()
df_perfil['cluster'] = etiquetas_cluster

# Calculamos el promedio de cada feature por cluster
# groupby('cluster'): agrupa filas por valor de cluster (0, 1, 2)
# .mean(): promedio de cada columna dentro de cada grupo
perfiles = df_perfil.groupby('cluster')[features_analisis].mean().round(3)

# Transponemos la tabla para que las features sean filas y los clusters columnas
# Esto facilita la lectura: vemos todas las características de cada cluster en columnas
print('=== Perfil promedio por cluster ===')
perfiles.T

In [ ]:
# pca_2d.components_: array de shape (2, 14)
# components_[0]: los 14 loadings del PC1 — cuánto contribuye cada feature al eje X
# components_[1]: los 14 loadings del PC2 — cuánto contribuye cada feature al eje Y
# Un loading positivo alto → cuando esa feature sube, el estudiante se mueve a la derecha/arriba
# Un loading negativo alto → cuando esa feature sube, el estudiante se mueve a la izquierda/abajo
loadings = pd.DataFrame(
    pca_2d.components_.T,                           # .T: features en filas, PCs en columnas
    index=df.drop(columns=['cluster']).columns,     # nombres de las 14 features como índice
    columns=['PC1', 'PC2']                          # nombres de los componentes
).round(3)

print('=== Loadings del PCA (contribución de cada feature a cada componente) ===')
print()
print('PC1 — las features que más definen el eje X del gráfico:')
print('  Positivo → esa feature mueve el punto hacia la DERECHA')
print('  Negativo → esa feature mueve el punto hacia la IZQUIERDA')
print()
# sort_values(ascending=False): de mayor contribución positiva a mayor contribución negativa
print(loadings['PC1'].sort_values(ascending=False).to_string())
print()
print('PC2 — las features que más definen el eje Y del gráfico:')
print('  Positivo → esa feature mueve el punto hacia ARRIBA')
print('  Negativo → esa feature mueve el punto hacia ABAJO')
print()
print(loadings['PC2'].sort_values(ascending=False).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel izquierdo: loadings de PC1 ─────────────────────────────────────────
# sort_values(): ordena de más negativo a más positivo para mejor lectura horizontal
loadings_pc1 = loadings['PC1'].sort_values()

# Color condicional: verde si contribuye positivamente, rojo si negativamente
colores_pc1 = ['#d73027' if v < 0 else '#4dac26' for v in loadings_pc1]

# Gráfico horizontal: features en eje Y, su loading en eje X
axes[0].barh(loadings_pc1.index, loadings_pc1.values,
             color=colores_pc1, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)   # línea en 0: separación positivo/negativo
axes[0].set_title(
    'Loadings del PC1
'
    'Verde = mueve al estudiante a la DERECHA | Rojo = mueve a la IZQUIERDA'
)
axes[0].set_xlabel('Loading (contribución al componente PC1)')

# ── Panel derecho: loadings de PC2 ───────────────────────────────────────────
loadings_pc2 = loadings['PC2'].sort_values()
colores_pc2  = ['#d73027' if v < 0 else '#4dac26' for v in loadings_pc2]

axes[1].barh(loadings_pc2.index, loadings_pc2.values,
             color=colores_pc2, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title(
    'Loadings del PC2
'
    'Verde = mueve al estudiante hacia ARRIBA | Rojo = mueve hacia ABAJO'
)
axes[1].set_xlabel('Loading (contribución al componente PC2)')

fig.suptitle(
    'Interpretación de los ejes del gráfico PCA
'
    '¿Qué hábito representa la posición horizontal (PC1)? ¿Y la vertical (PC2)?',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

print()
print('=== Interpretación de los ejes del gráfico PCA ===')
print()
top3_pc1_pos = loadings['PC1'].nlargest(3).index.tolist()
top3_pc1_neg = loadings['PC1'].nsmallest(3).index.tolist()
print(f'PC1 (eje X):')
print(f'  Estudiantes a la DERECHA del gráfico tienden a tener ALTOS:  {", ".join(top3_pc1_pos)}')
print(f'  Estudiantes a la IZQUIERDA del gráfico tienden a tener ALTOS: {", ".join(top3_pc1_neg)}')
print()
top3_pc2_pos = loadings['PC2'].nlargest(3).index.tolist()
top3_pc2_neg = loadings['PC2'].nsmallest(3).index.tolist()
print(f'PC2 (eje Y):')
print(f'  Estudiantes ARRIBA tienden a tener ALTOS:  {", ".join(top3_pc2_pos)}')
print(f'  Estudiantes ABAJO  tienden a tener ALTOS:  {", ".join(top3_pc2_neg)}')

In [ ]:
# Analizamos las features más diferenciadoras para cada cluster
# Comparamos el promedio del cluster vs el promedio global del dataset
print('=== Análisis de diferenciadores por cluster ===')
print()

media_global = df_perfil[features_analisis].mean()   # promedio de cada feature en todo el dataset

for k in range(K_OPTIMO):
    n_estudiantes = (etiquetas_cluster == k).sum()
    print(f'--- Cluster {k} ({n_estudiantes} estudiantes) ---')

    # Diferencia entre el perfil del cluster y la media global
    # Un valor positivo grande → esa feature es muy alta en este cluster
    # Un valor negativo grande → esa feature es muy baja en este cluster
    diferencias = perfiles.loc[k] - media_global

    # Las 3 features más altas en este cluster (los rasgos más positivos)
    top_altas = diferencias.nlargest(3)
    print(f'  Características altas:  {", ".join([f"{feat} (+{val:.2f})" for feat, val in top_altas.items()])}')

    # Las 3 features más bajas en este cluster (los rasgos más negativos)
    top_bajas = diferencias.nsmallest(3)
    print(f'  Características bajas:  {", ".join([f"{feat} ({val:.2f})" for feat, val in top_bajas.items()])}')
    print()

---
## 🔍 Sección 7 — La gran revelación: ¿K-Means encontró los aprobados y reprobados?

Este es el momento más importante de la clase.

**Recuerda:** K-Means nunca vio las etiquetas de `aprueba`. Agrupó a los estudiantes **solo basándose en sus hábitos**.

Ahora vamos a revelar si esos grupos coinciden con quienes realmente aprueban o reprueban.

Si hay una correspondencia fuerte, eso significa algo profundo: **los hábitos de un estudiante son suficientes para predecir su resultado académico**, incluso sin un modelo supervisado.

In [ ]:
# Agregamos las etiquetas reales (que ocultamos al inicio) al DataFrame de análisis
df_revelacion = df[['cluster']].copy()
df_revelacion['aprueba_real']   = aprueba_real.values     # etiqueta real: 0 o 1
df_revelacion['exam_score_real'] = exam_score_real.values  # puntaje real del examen

# ── Tabla cruzada: ¿qué proporción de cada cluster aprueba? ──────────────────
# pd.crosstab cuenta cuántos estudiantes de cada cluster tienen cada valor de aprueba_real
tabla_cruzada = pd.crosstab(
    df_revelacion['cluster'],     # filas: número de cluster (0, 1, 2)
    df_revelacion['aprueba_real'],# columnas: si aprueba (0 = no, 1 = sí)
    margins=True                  # agrega totales por fila y columna
)
tabla_cruzada.columns = ['Reprueba (0)', 'Aprueba (1)', 'Total']
tabla_cruzada.index   = [f'Cluster {i}' for i in range(K_OPTIMO)] + ['Total']

print('=== Tabla cruzada: Cluster vs Resultado real ===')
print(tabla_cruzada)
print()

# Proporción de aprobados dentro de cada cluster
print('=== Tasa de aprobación por cluster ===')
for k in range(K_OPTIMO):
    mask = df_revelacion['cluster'] == k
    tasa_aprobacion = df_revelacion.loc[mask, 'aprueba_real'].mean()
    puntaje_medio   = df_revelacion.loc[mask, 'exam_score_real'].mean()
    n               = mask.sum()
    print(f'  Cluster {k} ({n} estudiantes): '
          f'{tasa_aprobacion:.1%} aprueba | puntaje promedio: {puntaje_medio:.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Panel izquierdo: distribución de puntajes por cluster ────────────────────
# Si los clusters capturan diferencias reales, deberíamos ver distribuciones
# de puntaje bien separadas entre clusters
colores_cluster = ['#e74c3c', '#3498db', '#2ecc71']

for k in range(K_OPTIMO):
    mask = df_revelacion['cluster'] == k
    scores_k = df_revelacion.loc[mask, 'exam_score_real']

    # KDE: curva de densidad suavizada para ver la distribución de puntajes
    # Mejor que histogramas superpuestos porque es más fácil de leer con varios grupos
    scores_k.plot.kde(ax=axes[0], label=f'Cluster {k}',
                      color=colores_cluster[k], linewidth=2)

axes[0].axvline(60, color='black', linestyle='--', linewidth=1.5,
                label='Umbral de aprobación (60)')
axes[0].set_title(
    'Distribución de puntajes reales por cluster
'
    'Si los clusters están bien definidos, las curvas deben separarse'
)
axes[0].set_xlabel('Puntaje del examen (exam_score)')
axes[0].set_ylabel('Densidad')
axes[0].legend()

# ── Panel derecho: clusters en PCA, coloreados por aprueba/reprueba real ─────
# Comparamos el coloreado por cluster (lo que encontró K-Means)
# vs el coloreado por la realidad (aprueba=0/1)
# Si son similares → K-Means descubrió la estructura real sin ver las etiquetas
colores_real = ['tomato' if a == 0 else 'steelblue'
                for a in df_revelacion['aprueba_real']]

axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                c=colores_real, alpha=0.5, s=20, edgecolors='white', linewidths=0.2)

# Leyenda manual para la gráfica
from matplotlib.patches import Patch
leyenda = [Patch(color='tomato',    label='Reprueba (real)'),
           Patch(color='steelblue', label='Aprueba (real)')]
axes[1].legend(handles=leyenda)

axes[1].set_title(
    'Mismos estudiantes en PCA 2D
coloreados por resultado REAL (aprueba/reprueba)
'
    'Compara con el gráfico de clusters para ver la correspondencia'
)
axes[1].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} varianza)')
axes[1].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} varianza)')

plt.tight_layout()
plt.show()

---
## 🏋️ Sección 8 — Ejercicio resuelto

### Planteamiento

El análisis de PCA nos da los **loadings**: cuánto contribuye cada feature original a cada componente principal. Esto nos permite interpretar qué significa PC1 y PC2 en términos de hábitos estudiantiles.

**Tu tarea:**
1. Extraer los loadings del PCA entrenado
2. Identificar qué features tienen mayor carga (positiva o negativa) en PC1 y PC2
3. Interpretar qué representa cada componente en términos de comportamiento estudiantil
4. Usar esa interpretación para explicar la posición de los clusters en el gráfico PCA

In [ ]:
# ── PASO 1: Extraer los loadings del PCA ─────────────────────────────────────
# pca_2d.components_: array (2, 14) con los loadings de cada componente
# components_[0]: loadings del PC1 (cuánto contribuye cada feature al eje X del gráfico)
# components_[1]: loadings del PC2 (cuánto contribuye cada feature al eje Y del gráfico)
# Un loading positivo alto → cuando esa feature sube, el estudiante se va a la derecha/arriba
# Un loading negativo alto → cuando esa feature sube, el estudiante se va a la izquierda/abajo
loadings = pd.DataFrame(
    pca_2d.components_.T,          # transponemos: features en filas, PCs en columnas
    index=df.drop(columns=['cluster']).columns,   # nombres de las features como índice
    columns=['PC1', 'PC2']         # nombres de los componentes
).round(3)

# ── PASO 2: Identificar las features más influyentes en PC1 ───────────────────
print('=== Loadings de PC1 (las features que más definen el eje X) ===')
print('Positivo = esa feature mueve a los estudiantes hacia la DERECHA del gráfico')
print('Negativo = esa feature mueve a los estudiantes hacia la IZQUIERDA')
print()
print(loadings['PC1'].sort_values(ascending=False).to_string())

print()
print('=== Loadings de PC2 (las features que más definen el eje Y) ===')
print(loadings['PC2'].sort_values(ascending=False).to_string())

In [ ]:
# ── PASO 3 y 4: Visualizar loadings e interpretar ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Panel izquierdo: loadings de PC1 ─────────────────────────────────────────
loadings_pc1 = loadings['PC1'].sort_values()   # ordena de más negativo a más positivo
colores_pc1  = ['#d73027' if v < 0 else '#4dac26' for v in loadings_pc1]   # rojo/verde

axes[0].barh(loadings_pc1.index, loadings_pc1.values, color=colores_pc1, edgecolor='white')
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title(
    'Loadings de PC1
Verde=mueve a derecha | Rojo=mueve a izquierda'
)
axes[0].set_xlabel('Loading (contribución al PC1)')

# ── Panel derecho: loadings de PC2 ───────────────────────────────────────────
loadings_pc2 = loadings['PC2'].sort_values()
colores_pc2  = ['#d73027' if v < 0 else '#4dac26' for v in loadings_pc2]

axes[1].barh(loadings_pc2.index, loadings_pc2.values, color=colores_pc2, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title(
    'Loadings de PC2
Verde=mueve hacia arriba | Rojo=mueve hacia abajo'
)
axes[1].set_xlabel('Loading (contribución al PC2)')

fig.suptitle(
    'Interpretación de los ejes PCA:
'
    '¿Qué hábito representa el eje X (PC1)? ¿Y el eje Y (PC2)?',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

print()
print('=== Interpretación de los componentes principales ===')
print()
top_pc1_pos = loadings['PC1'].nlargest(3).index.tolist()
top_pc1_neg = loadings['PC1'].nsmallest(3).index.tolist()
print(f'PC1 (eje X del gráfico):')
print(f'  Valores altos de PC1 (derecha) asociados con: {", ".join(top_pc1_pos)}')
print(f'  Valores bajos de PC1 (izquierda) asociados con: {", ".join(top_pc1_neg)}')
print()
top_pc2_pos = loadings['PC2'].nlargest(3).index.tolist()
top_pc2_neg = loadings['PC2'].nsmallest(3).index.tolist()
print(f'PC2 (eje Y del gráfico):')
print(f'  Valores altos de PC2 (arriba) asociados con: {", ".join(top_pc2_pos)}')
print(f'  Valores bajos de PC2 (abajo) asociados con: {", ".join(top_pc2_neg)}')

---
## 🏁 Resumen de la clase

### Todo lo que construiste hoy

```
Dataset sin etiquetas (quitamos exam_score y aprueba)
    │
    ├─ PREPARACIÓN PARA CLUSTERING ────────────────────────────────────────────
    │    ├── Encoding de categóricas (igual que clases anteriores)
    │    ├── StandardScaler sobre TODOS los datos (sin train/test split)
    │    └── Por qué el escalado es CRÍTICO para K-Means (distancias euclidianas)
    │
    ├─ K-MEANS CLUSTERING ─────────────────────────────────────────────────────
    │    ├── Algoritmo: inicializar → asignar → recalcular → converger
    │    ├── Método del Codo: inercia vs K para elegir el número de clusters
    │    ├── Silhouette Score: medir qué tan bien definidos están los clusters
    │    ├── KMeans(n_clusters=K, n_init=10).fit_predict(X_scaled)
    │    └── Análisis de perfiles: qué caracteriza a cada cluster
    │
    ├─ PCA ────────────────────────────────────────────────────────────────────
    │    ├── Por qué necesitamos PCA: no podemos ver 14 dimensiones
    │    ├── Varianza explicada por componente y acumulada
    │    ├── PCA(n_components=2).fit_transform(X_scaled) → espacio 2D
    │    ├── Visualización de clusters en 2D
    │    └── Loadings: qué features definen cada componente principal
    │
    └─ LA GRAN REVELACIÓN ─────────────────────────────────────────────────────
         └── ¿Los clusters de K-Means corresponden con aprueba/reprueba?
             → Si sí: los hábitos solos predicen el resultado académico
             → Validación cruzada entre clustering y etiquetas reales
```

---

### Conceptos clave

| Concepto | Definición en una frase |
|---|---|
| **Clustering** | Agrupar datos similares sin saber de antemano cuántos grupos hay ni cuáles son |
| **K-Means** | Algoritmo que asigna cada punto al centroide más cercano y refina los centroides iterativamente |
| **Inercia** | Suma de distancias al cuadrado dentro de cada cluster: mide qué tan compactos son |
| **Silhouette Score** | Mide qué tan bien separados están los clusters entre sí (+1 = perfecto, -1 = clusters mezclados) |
| **PCA** | Comprime N dimensiones en M componentes que capturan la mayor varianza posible |
| **Varianza explicada** | % de información del dataset original capturada por los componentes seleccionados |
| **Loading** | Cuánto contribuye cada feature original a cada componente principal |

---

### 🧠 Preguntas para reflexionar

1. K-Means requiere que especifiques K antes de entrenar. ¿Qué harías si el Método del Codo no muestra un "codo" claro?
2. ¿Por qué el escalado con StandardScaler es **obligatorio** para K-Means pero **opcional** para un Árbol de Decisión?
3. Si la tasa de aprobación dentro de cada cluster es muy similar entre clusters (ej: 70%, 72%, 74%), ¿qué nos dice eso sobre la utilidad del clustering para este problema?
4. PCA "comprimió" 14 features en 2 componentes. ¿Siempre perdemos información con PCA? ¿Cuándo podríamos perder muy poco?
5. Si un directivo te pregunta "¿por qué el Cluster 0 tiene peor rendimiento académico?", ¿cómo usarías el heatmap de perfiles y los loadings de PCA para responderle de forma concreta y accionable?